# 🚀 TARS v3 — Gelişmiş RAG + QLoRA Pipeline

## v2'den farklar
| Bileşen | v2 | v3 |
|---------|----|----||
| Chunking | Sabit 300 karakter | Semantik (cümle sınırına uyar) |
| Retrieval | Sadece FAISS cosine | FAISS + BM25 hibrit |
| Reranking | Yok | CrossEncoder ile yeniden sıralama |
| Bellek | Yok (tek turlu) | ConversationChain (geçmiş tutulur) |
| Kalite | Basit grounding skoru | ROUGE-L + kaynak doğrulama |
| Eğitim | Temel SFT | Packing + gradient checkpointing |

## 📦 1. Kurulum

In [79]:
!pip install -q transformers peft accelerate datasets bitsandbytes trl
!pip install -q sentence-transformers faiss-cpu
!pip install -q python-docx pypdf
!pip install -q rank-bm25 rouge-score
# rank-bm25  → BM25 kelime eşleştirme (hibrit retrieval için)
# rouge-score → ROUGE-L metriği (cevap kalitesi ölçümü için)
print('✅ Kurulum tamamlandı')

✅ Kurulum tamamlandı


In [80]:
import torch, json, os, glob, re
import numpy as np
from pathlib import Path

import torch, gc


device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'✅ GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('⚠️  GPU bulunamadı')

✅ GPU : Tesla T4
   VRAM: 15.6 GB


## ⚙️ 2. Ayarlar

In [81]:
JSONL_DIR    = '/content/jsonl'
DOCS_DIR     = '/content/docs'
OUTPUT_DIR   = '/content/tars_v3'

LLM_MODEL    = 'Qwen/Qwen2.5-1.5B-Instruct' # Modeli 3B'den 1.5B'ye düşürüldü
EMBED_MODEL  = 'paraphrase-multilingual-MiniLM-L12-v2'
# CrossEncoder: iki metni birlikte okuyarak daha hassas skor üretir
# v2'de yoktu — retrieval kalitesini önemli ölçüde artırır
RERANK_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

# RAG
CHUNK_SIZE    = 500    # v2'de 300'dü, semantik bölücüyle biraz büyütebiliriz
CHUNK_OVERLAP = 80
TOP_K         = 6     # BM25+FAISS'ten 6 al, reranking sonrası 3'e indir
FINAL_K       = 3     # Modele gönderilecek nihai chunk sayısı
BM25_WEIGHT   = 0.3   # Hibrit skorda BM25'in ağırlığı (FAISS: 0.7)

# Fine-tuning
MAX_LENGTH   = 512
BATCH_SIZE   = 1
GRAD_ACCUM   = 16
EPOCHS       = 4
LR           = 2e-4
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROP    = 0.05

os.makedirs(JSONL_DIR, exist_ok=True)
os.makedirs(DOCS_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR,exist_ok=True)
print('✅ Ayarlar tamam')

✅ Ayarlar tamam


## 📂 3. Veri Yükleme

In [82]:
# JSONL
raw_data = []
jsonl_files = list(Path(JSONL_DIR).glob('*.jsonl'))

if not jsonl_files:
    print('⚠️  JSONL bulunamadı — demo veri oluşturuluyor...')
    demo = [
        {"instruction": "ADC biriminin roket aviyonik sistemindeki rolü nedir?",
         "input": "Veri dönüştürme",
         "output": "Analog sensörlerden gelen voltaj değerlerini uçuş bilgisayarının işleyebileceği dijital verilere dönüştürür."},
        {"instruction": "IMU sensörünün uçuş kontrol sistemindeki işlevi nedir?",
         "input": "",
         "output": "İvme, açısal hız ve manyetik alan verilerini ölçerek roketin oryantasyonunu ve hareketini belirler."},
        {"instruction": "UART protokolü nedir?", "input": "Haberleşme",
         "output": "Asenkron seri haberleşme protokolüdür. TX ve RX hatları üzerinden veri iletişimi sağlar."},
        {"instruction": "SPI ile I2C arasındaki fark nedir?", "input": "",
         "output": "SPI 4 hat kullanır ve daha hızlıdır. I2C 2 hat kullanır ve daha az pin tüketir."},
        {"instruction": "PWM sinyali ne için kullanılır?", "input": "Motor kontrol",
         "output": "Motor hızı, servo konumu veya LED parlaklığını darbe genişliği ile kontrol eder."},
        {"instruction": "Watchdog timer nedir?", "input": "",
         "output": "Sistemin donmasını önlemek için belirli aralıklarla sıfırlanması gereken donanım sayacıdır."},
        {"instruction": "DMA ne zaman kullanılır?", "input": "Bellek",
         "output": "CPU müdahalesi olmadan bellek ile çevre birimler arasında veri aktarımı sağlar."},
        {"instruction": "GPIO pini nasıl yapılandırılır?", "input": "",
         "output": "Input veya output olarak ayarlanır. Pull-up/pull-down dirençleri varsayılan durumu belirler."},
        {"instruction": "CAN bus neden tercih edilir?", "input": "Otomotiv",
         "output": "Gürültüye dayanıklılığı ve çok düğümlü yapısıyla endüstriyel uygulamalarda yaygın kullanılır."},
        {"instruction": "RTOS kullanmanın avantajları nelerdir?", "input": "",
         "output": "Gerçek zamanlı görev planlaması ve deterministik yanıt süresi sağlar."},
    ]
    with open(f'{JSONL_DIR}/demo.jsonl', 'w', encoding='utf-8') as f:
        for item in demo:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    jsonl_files = [Path(f'{JSONL_DIR}/demo.jsonl')]

for path in jsonl_files:
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try: raw_data.append(json.loads(line))
                except: pass

print(f'✅ JSONL kayıt: {len(raw_data)}')

✅ JSONL kayıt: 380


In [83]:
# PDF + DOCX yükle
def read_pdf(path):
    try:
        from pypdf import PdfReader
        return '\n'.join(p.extract_text() or '' for p in PdfReader(path).pages)
    except Exception as e:
        print(f'  ⚠️  {Path(path).name}: {e}'); return ''

def read_docx(path):
    try:
        from docx import Document
        return '\n'.join(p.text for p in Document(path).paragraphs if p.text.strip())
    except Exception as e:
        print(f'  ⚠️  {Path(path).name}: {e}'); return ''

documents = []
for path in glob.glob(f'{DOCS_DIR}/**/*.pdf', recursive=True):
    text = read_pdf(path)
    if text.strip():
        documents.append({'title': Path(path).stem, 'text': text, 'source': Path(path).name})
        print(f'  📄 PDF : {Path(path).name} ({len(text):,} kar.)')

for path in glob.glob(f'{DOCS_DIR}/**/*.docx', recursive=True):
    text = read_docx(path)
    if text.strip():
        documents.append({'title': Path(path).stem, 'text': text, 'source': Path(path).name})
        print(f'  📄 DOCX: {Path(path).name} ({len(text):,} kar.)')

# JSONL → doküman
for item in raw_data:
    documents.append({
        'title':  item['instruction'][:60],
        'text':   f"Soru: {item['instruction']}\nCevap: {item['output']}",
        'source': 'jsonl'
    })

print(f'\n✅ Toplam doküman: {len(documents)}')

  📄 PDF : STM32 ile Mikrodenetleyici Dünyasına Giriş_ Donanım ve Çalışma Mantığı Rehberi.pdf (8,928 kar.)
  📄 PDF : Mega2650PRO ve BMP280 Entegrasyonu_ Sistem Kapasite ve Performans Analiz Raporu.pdf (9,141 kar.)
  📄 PDF : Gökyüzünü Verilerle Okumak_ BMP280 ve Mega2650PRO ile İrtifa Rehberi.pdf (8,202 kar.)
  📄 PDF : Embedded_system_exercise.pdf (52,201 kar.)
  📄 PDF : Dev Güç, Mikro Boyut_ Yeni Nesil Gömülü Sistemlerden Öğrendiğimiz 5 Şaşırtıcı Gerçek.pdf (6,888 kar.)
  📄 PDF : Gömülü Sistemlerde Standart ve Ağ Protokolü Uyumluluk Analizi.pdf (9,380 kar.)
  📄 PDF : Elektronik Dünyasına İlk Adım_ Mikrodenetleyiciler ve Sensörlerin Gizemli Dünyası.pdf (8,067 kar.)
  📄 PDF : C++_Mastery_Book.pdf (1,034,702 kar.)
  📄 PDF : 2022_İtü.pdf (54,061 kar.)
  📄 PDF : Mega2650PRO ve BMP280 Endüstriyel Entegrasyon ve Prototipleme Rehberi.pdf (8,525 kar.)
  📄 PDF : Gömülü Sistemlerin Dünyası_ Kavramsal Temeller Rehberi.pdf (9,024 kar.)
  📄 PDF : Model Roketçilik Roketsan.pdf (114,649 kar.)
  📄 PDF :

## 🔪 4. Semantik Chunking (YENİ)

**v2'de ne vardı?**  
Sabit 300 karakterde kes — cümlenin ortasında bile olsa.

**v3'te ne var?**  
Önce nokta/soru işareti/ünlem ile cümlelere böl, sonra cümleleri CHUNK_SIZE'ı aşmadan birleştir.  
Cümle asla ortadan kesilmez → retrieval sırasında anlam bütünlüğü korunur.

In [84]:
def semantic_chunk(text, max_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Cümle sınırlarına saygı duyan akıllı metin bölücü.
    Adımlar:
      1. Metni cümlelere böl (nokta/soru/ünlem ile biten)
      2. Cümleleri max_size'ı aşmadan birleştir
      3. Örtüşme için önceki chunk'ın son cümlesini ekle
    """
    # Cümle bölme: nokta, soru işareti, ünlem + boşluk/satır sonu
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current = []
    current_len = 0

    for sent in sentences:
        sent_len = len(sent)
        # Mevcut gruba sığıyorsa ekle
        if current_len + sent_len <= max_size:
            current.append(sent)
            current_len += sent_len + 1
        else:
            if current:
                chunks.append(' '.join(current))
            # Overlap: önceki chunk'ın son cümlesini yeni chunk'a taşı
            if current and len(current[-1]) <= overlap:
                current = [current[-1], sent]
                current_len = len(current[-1]) + sent_len + 1
            else:
                current = [sent]
                current_len = sent_len

    if current:
        chunks.append(' '.join(current))

    return [c for c in chunks if len(c) > 20]  # Çok kısa parçaları at


all_chunks = []
for doc in documents:
    for chunk in semantic_chunk(doc['text']):
        all_chunks.append({
            'text':   chunk,
            'source': doc['source'],
            'title':  doc['title'],
        })

lengths = [len(c['text']) for c in all_chunks]
print(f'✅ Semantik chunk sayısı: {len(all_chunks)}')
print(f'   Min: {min(lengths)} | Ort: {np.mean(lengths):.0f} | Max: {max(lengths)} karakter')

✅ Semantik chunk sayısı: 3278
   Min: 22 | Ort: 473 | Max: 18496 karakter


## 🔍 5. Hibrit Retrieval: FAISS + BM25 (YENİ)

**v2'de ne vardı?**  
Sadece FAISS (dense / anlamsal benzerlik).  
Sorun: "UART baud rate" gibi spesifik teknik terimler için kelime eşleştirmesi daha güvenilirdir.

**v3'te ne var?**  
- FAISS: anlam benzerliği ("nasıl çalışır" tipi sorular)
- BM25: kelime eşleştirme ("X nedir" tipi spesifik terimler)
- İkisinin skoru ağırlıklı ortalamayla birleştirilir

In [85]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from rank_bm25 import BM25Okapi

# ── Dense embedding (FAISS için) ──────────────────────────────────────────
print(f'Embedding modeli: {EMBED_MODEL}')
embed_model = SentenceTransformer(EMBED_MODEL)

chunk_texts = [c['text'] for c in all_chunks]
embeddings  = embed_model.encode(
    chunk_texts, normalize_embeddings=True,
    show_progress_bar=True, batch_size=64
).astype(np.float32)

EMBED_DIM = embeddings.shape[1]
index     = faiss.IndexFlatIP(EMBED_DIM)
index.add(embeddings)
print(f'✅ FAISS: {index.ntotal} vektör ({EMBED_DIM}D)')

# ── Sparse (BM25 için) ────────────────────────────────────────────────────
# BM25 kelime bazlı çalışır — Türkçe için basit tokenize
def tokenize_tr(text):
    return text.lower().split()

tokenized_corpus = [tokenize_tr(c) for c in chunk_texts]
bm25 = BM25Okapi(tokenized_corpus)
print(f'✅ BM25 indeksi: {len(tokenized_corpus)} döküman')

# ── CrossEncoder (reranking için) ─────────────────────────────────────────
# CrossEncoder iki metni aynı anda okur → daha hassas alaka skoru
# Ama yavaştır — bu yüzden sadece top-K adayına uygulanır
print(f'CrossEncoder: {RERANK_MODEL}')
cross_encoder = CrossEncoder(RERANK_MODEL)
print('✅ CrossEncoder hazır')

Embedding modeli: paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/52 [00:00<?, ?it/s]

✅ FAISS: 3278 vektör (384D)
✅ BM25 indeksi: 3278 döküman
CrossEncoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CrossEncoder hazır


In [86]:
def hybrid_retrieve(query, k=TOP_K, final_k=FINAL_K, alpha=BM25_WEIGHT):
    """
    3 aşamalı retrieval:
      1. FAISS dense search  → top-k aday
      2. BM25 sparse search  → top-k aday
      3. Skorları birleştir → CrossEncoder ile rerank → final_k döndür

    alpha: BM25 ağırlığı (0=sadece FAISS, 1=sadece BM25)
    """
    n = len(all_chunks)

    # ── 1. FAISS ─────────────────────────────────────────────────────────
    q_emb  = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    f_scores, f_idx = index.search(q_emb, min(k*2, n))
    faiss_scores = np.zeros(n)
    for score, idx in zip(f_scores[0], f_idx[0]):
        faiss_scores[idx] = float(score)

    # ── 2. BM25 ──────────────────────────────────────────────────────────
    bm25_raw = bm25.get_scores(tokenize_tr(query))
    # Normalize et [0,1]
    bm25_max = bm25_raw.max() + 1e-9
    bm25_scores = bm25_raw / bm25_max

    # ── 3. Hibrit skor ───────────────────────────────────────────────────
    # FAISS zaten [0,1] (normalize vektörler)
    hybrid = (1 - alpha) * faiss_scores + alpha * bm25_scores

    # Top-k adayı seç
    top_idx = np.argsort(hybrid)[::-1][:k]
    candidates = [(all_chunks[i], float(hybrid[i])) for i in top_idx]

    # ── 4. CrossEncoder Reranking ─────────────────────────────────────────
    # CrossEncoder her (query, chunk) çiftini birlikte değerlendirir
    # Bu, dense embedding'in kaçırdığı nüansları yakalar
    pairs  = [(query, c['text']) for c, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)

    # CrossEncoder skoru + hibrit skor → nihai sıralama
    ranked = sorted(
        zip(candidates, ce_scores),
        key=lambda x: x[1],
        reverse=True
    )[:final_k]

    results = []
    for (chunk, hybrid_score), ce_score in ranked:
        results.append({
            'text':         chunk['text'],
            'source':       chunk['source'],
            'title':        chunk['title'],
            'hybrid_score': round(hybrid_score, 3),
            'ce_score':     round(float(ce_score), 3),
        })
    return results


# Test
test_r = hybrid_retrieve('ADC birimi ne işe yarar?')
print('── Hibrit retrieval testi ──')
for r in test_r:
    print(f'  hybrid={r["hybrid_score"]} ce={r["ce_score"]} | {r["source"]} → {r["text"][:60]}...')

── Hibrit retrieval testi ──
  hybrid=0.418 ce=1.134 | jsonl → Soru: ADC (Analog-to-Digital Converter) biriminin roket aviy...
  hybrid=0.354 ce=-5.804 | C++_Mastery_Book.pdf → 74 │   // TR: ADC dönüşüm tamamlandı kesmesini simüle eder. ...
  hybrid=0.345 ce=-5.89 | STM32 ile Mikrodenetleyici Dünyasına Giriş_ Donanım ve Çalışma Mantığı Rehberi.pdf → ●  AVDD:   ADC  ve  analog  devreler  için  gereken   temiz ...


## 🤖 6. QLoRA Fine-Tuning (İyileştirilmiş)

**v2'den farklar:**
- `packing=True` → kısa örnekleri birleştir, boş token olmadan eğit (daha hızlı)
- `gradient_checkpointing=True` → aktivasyonları yeniden hesapla, VRAM tasarrufu
- `group_by_length=True` → benzer uzunlukları aynı batch'e koy, padding azalır

In [87]:
# Veri formatlama (v2 ile aynı)
# 6. QLoRA Fine-Tuning bölümündeki ilgili hücre
SYSTEM_PROMPT = (
    "Sen bir teknik asistansın. SADECE sana verilen kaynaklara dayanarak cevap ver. "
    "Eğer cevap kaynaklarda yoksa 'Bu bilgi kaynaklarımda bulunmuyor' de ve uydurma. "
    "Cevaplarını teknik terimler dışında tamamen Türkçe kur."
)

def format_sample(item):
    # Bu fonksiyon artık yeni SYSTEM_PROMPT'u kullanarak eğitim verisini hazırlayacak
    inp     = item.get('input', '').strip()
    user    = f"{item['instruction']}\\nBağlam: {inp}" if inp else item['instruction']
    return {'text': (
        f'<|im_start|>system\\n{SYSTEM_PROMPT}<|im_end|>\\n'
        f'<|im_start|>user\\n{user}<|im_end|>\\n'
        f'<|im_start|>assistant\\n{item["output"]}<|im_end|>'
    )}

from datasets import Dataset
formatted = [format_sample(d) for d in raw_data]
np.random.seed(42)
idx   = np.random.permutation(len(formatted))
split = max(1, int(len(formatted) * 0.9))
train_dataset = Dataset.from_list([formatted[i] for i in idx[:split]])
val_dataset   = Dataset.from_list([formatted[i] for i in idx[split:]])
print(f'✅ Train: {len(train_dataset)}  Val: {len(val_dataset)}')

✅ Train: 342  Val: 38


In [88]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True,
)

# YENİ: gradient checkpointing — VRAM'i ~%30 azaltır
# Aktivasyonları forward pass'te saklamak yerine backward'da yeniden hesaplar
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=LORA_DROP, bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [89]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    fp16                        = True,
    logging_steps               = 5,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'none',

    gradient_checkpointing      = True,
    optim                       = "paged_adamw_32bit",

    # max_seq_length was causing issues here, removed to avoid TypeError
    dataset_text_field          = 'text',
    # YENİ: packing — kısa örnekleri birleştirerek padding'i azaltır
    # Her batch'te daha fazla gerçek token → daha verimli eğitim
    packing                     = True,
    # YENİ: benzer uzunluktaki örnekleri aynı batch'e koy
    # Padding miktarını azaltır → hız artar
    group_by_length             = True,
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    processing_class=tokenizer
    # max_seq_length will now be handled implicitly by SFTTrainer or tokenizer
)

print('🚀 Eğitim başlıyor...')
train_result = trainer.train()
print(f'✅ Loss: {train_result.training_loss:.4f}')

adapter_path = f'{OUTPUT_DIR}/final_adapter'
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'✅ Adaptör kaydedildi: {adapter_path}')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/342 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/342 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/342 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/38 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/38 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/38 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Eğitim başlıyor...


OutOfMemoryError: CUDA out of memory. Tried to allocate 594.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 567.81 MiB is free. Including non-PyTorch memory, this process has 14.01 GiB memory in use. Of the allocated memory 12.17 GiB is allocated by PyTorch, and 1.70 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 💬 7. Birleşik Sohbet: RAG + LLM + Bellek (YENİ)

**v2'de ne vardı?**  
Her soru bağımsız — model önceki soruyu hatırlamıyor.

**v3'te ne var?**  
`conversation_history` listesi tutulur. Her turda önceki sorular ve cevaplar  
prompt'a eklenir. Model "az önce ADC sordun, şimdi bunu soruyorsun" bağlamını görür.

In [ ]:
from peft import PeftModel

adapter_path = f'{OUTPUT_DIR}/final_adapter'
print('Model yükleniyor...')
inf_tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
inf_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, torch_dtype=torch.float16,
    device_map='auto', trust_remote_code=True,
)
inf_model = PeftModel.from_pretrained(inf_model, adapter_path)
inf_model.eval()
print('✅ Model hazır')

In [ ]:
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def hallucination_check(answer, retrieved_chunks):
    """
    Cevabın kaynaklara ne kadar dayandığını ölç.
    ROUGE-L: cevap ile bağlam arasındaki en uzun ortak alt-dizi.
    v2'deki basit kelime örtüşmesinden daha güvenilir.
    """
    context = ' '.join(r['text'] for r in retrieved_chunks)
    score   = rouge.score(context, answer)['rougeL'].fmeasure

    if score >= 0.4:
        flag = '✅ Güvenilir'
    elif score >= 0.2:
        flag = '⚠️  Kısmen kaynaklı'
    else:
        flag = '❌ Hallucination riski'

    return round(score, 3), flag


# Sohbet geçmişi — oturum boyunca korunur
conversation_history = []


def tars_chat(soru, max_new_tokens=256, temperature=0.3, max_history=4):
    global conversation_history

    retrieved = hybrid_retrieve(soru)
    context_parts = [f'[Kaynak {i+1} — {r["source"]}]\\n{r["text"]}'
                     for i, r in enumerate(retrieved)]
    context = '\\n\\n'.join(context_parts)

    # BURAYI GÜNCELLE:
    system = (
        "Sen bir teknik asistansın. SADECE sana verilen kaynaklara dayanarak cevap ver. "
        "Eğer cevap kaynaklarda yoksa 'Bu bilgi kaynaklarımda bulunmuyor' de ve uydurma. "
        "Cevaplarını teknik terimler dışında tamamen Türkçe kur.\\n\\n"
        f"KULLANILACAK KAYNAKLAR:\\n{context}"
    )

    # ... fonksiyonun geri kalanı aynı kalacak ...

    # ── 3. Sohbet geçmişini ekle (bellek) ────────────────────────────────
    # Son max_history turu al — çok eski geçmişi kes (token limiti)
    history_msgs = ''
    for turn in conversation_history[-max_history:]:
        history_msgs += (
            f'<|im_start|>user\n{turn["user"]}<|im_end|>\n'
            f'<|im_start|>assistant\n{turn["assistant"]}<|im_end|>\n'
        )

    # ── 4. Prompt ────────────────────────────────────────────────────────
    prompt = (
        f'<|im_start|>system\n{system}<|im_end|>\n'
        f'{history_msgs}'
        f'<|im_start|>user\n{soru}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )

    # ── 5. Üretim ────────────────────────────────────────────────────────
    inputs = inf_tokenizer(prompt, return_tensors='pt').to(inf_model.device)
    if inputs['input_ids'].shape[1] > MAX_LENGTH - 50:
        print('  ⚠️  Prompt uzun, geçmiş kısaltıldı.')

    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            do_sample          = temperature > 0,
            repetition_penalty = 1.1,
            eos_token_id       = inf_tokenizer.convert_tokens_to_ids('<|im_end|>'),
            pad_token_id       = inf_tokenizer.pad_token_id,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    answer = inf_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # ── 6. Hallucination kontrolü ─────────────────────────────────────────
    rouge_score, flag = hallucination_check(answer, retrieved)

    # ── 7. Geçmişe ekle ──────────────────────────────────────────────────
    conversation_history.append({'user': soru, 'assistant': answer})

    return {
        'answer':    answer,
        'sources':   [r['source'] for r in retrieved],
        'ce_scores': [r['ce_score'] for r in retrieved],
        'rouge_l':   rouge_score,
        'flag':      flag,
        'turn':      len(conversation_history),
    }

print('✅ tars_chat() hazır')

In [ ]:
# ── Test: Çok turlu sohbet ────────────────────────────────────────────────
# Bellek testi: ikinci soruda birinci soruya atıfta bulunuluyor

conversation_history = []  # Geçmişi sıfırla

test_questions = [
    'ADC biriminin roket aviyonik sistemindeki rolü nedir?',
    'Peki bu birim hangi sensörlerle birlikte çalışır?',  # Bellek testi
    'IMU sensörü hangi verileri ölçer?',
    'Watchdog timer neden kullanılır?',
]

print('=' * 65)
print('  TARS v3 — RAG + Bellek + Hallucination Guard')
print('=' * 65)

for soru in test_questions:
    print(f'\n❓ Tur {len(conversation_history)+1}: {soru}')
    result = tars_chat(soru)
    print(f'🤖 {result["answer"]}')
    print(f'📚 Kaynaklar     : {set(result["sources"])}')
    print(f'🎯 CE skorları   : {result["ce_scores"]}')
    print(f'📊 ROUGE-L       : {result["rouge_l"]}  {result["flag"]}')
    print('─' * 55)

In [ ]:
# ── İnteraktif sohbet ─────────────────────────────────────────────────────
# conversation_history burada KORUNUR — oturum boyunca geçmiş tutulur
# Sıfırlamak için: conversation_history = []

print(f'TARS v3 hazır. Geçmiş: {len(conversation_history)} tur')
print('Sıfırlamak için: conversation_history = []')
print('Çıkmak için: q\n')

while True:
    soru = input(f'[Tur {len(conversation_history)+1}] Siz: ').strip()
    if soru.lower() in ('q', 'quit', 'exit', ''):
        print('Görüşmek üzere!')
        break
    result = tars_chat(soru, verbose=False) if 'verbose' in tars_chat.__code__.co_varnames else tars_chat(soru)
    print(f'\nTARS: {result["answer"]}')
    print(f'[{result["flag"]} | ROUGE-L: {result["rouge_l"]} | Kaynaklar: {set(result["sources"])}]\n')